# NLI extraction

In [ ]:
# setup and loading the data

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
 
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
DATA_FILENAME = "books_big.parquet"
candidates = list(Path("/kaggle/input").rglob(DATA_FILENAME))
if not candidates:
    print("Available parquet files under /kaggle/input:")
    for p in Path("/kaggle/input").rglob("*.parquet"):
        print(" ", p)
    raise FileNotFoundError(f"{DATA_FILENAME} not found under /kaggle/input")
 
DATA_PATH = candidates[0]
print(f"Using: {DATA_PATH}")
 
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
MAX_LENGTH = 512
BATCH_SIZE = 64
CONFIDENCE_THRESHOLD = 0.7
MARGIN_THRESHOLD = 0.3
 
ASPECT_HYPOTHESES = {
    "plot": {
        "positive": "This review is positive about the plot.",
        "negative": "This review is negative about the plot.",
    },
    "characters": {
        "positive": "This review is positive about the characters.",
        "negative": "This review is negative about the characters.",
    },
    "writing": {
        "positive": "This review is positive about the writing style.",
        "negative": "This review is negative about the writing style.",
    },
}
ASP_NAMES = list(ASPECT_HYPOTHESES.keys())
ASP_COLS = [f"nli_aspect_{asp}" for asp in ASP_NAMES]
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print(f"Device: {DEVICE}")
print(f"Batch: {BATCH_SIZE}")
print(f"Threshold: conf >= {CONFIDENCE_THRESHOLD}, margin >= {MARGIN_THRESHOLD}")
 
df = pd.read_parquet(DATA_PATH)
print(f"\nRows: {len(df):,} | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
 
RATING_COL = "rating" if "rating" in df.columns else ("overall" if "overall" in df.columns else None)
print(f"Rating column: {RATING_COL}")
 
df = df.sort_values(["user_id", "timestamp"]).copy()
df["_rank"] = df.groupby("user_id")["timestamp"].rank(method="first", ascending=True)
df["_size"] = df.groupby("user_id")["user_id"].transform("count")
train_mask = ~((df["_rank"] == df["_size"]) | (df["_rank"] == df["_size"] - 1))
df = df.drop(columns=["_rank", "_size"])
 
train_texts = df.loc[train_mask, "text_combined"].fillna("").astype(str)
nonempty_mask = train_texts.str.strip().ne("")
valid_idx = train_texts[nonempty_mask].index
texts = train_texts.loc[valid_idx].tolist()
n = len(texts)
 
print(f"Train: {train_mask.sum():,} | Non-empty: {n:,}")
 

In [2]:
# model
 
print(f"Loading {NLI_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
model.to(DEVICE).eval()
 
label2id = {str(k).lower(): v for k, v in model.config.label2id.items()}
ENTAILMENT_ID = label2id["entailment"]
print(f"Entailment ID: {ENTAILMENT_ID}")
 
def score_batch(premises, hypothesis):
    enc = tokenizer(premises, [hypothesis]*len(premises),
                    padding=True, truncation="only_first",
                    max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)
    return probs[:, ENTAILMENT_ID].cpu().numpy().astype(np.float32)
 
print("\nSanity check:")
test_premise = "The plot was gripping, but the characters felt shallow."
for hyp in ["positive about the plot", "negative about the characters", "positive about the characters"]:
    full_hyp = f"This review is {hyp}."
    enc = tokenizer(test_premise, full_hyp, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy()
    print(f"  '{hyp}' → {p[ENTAILMENT_ID]:.4f}")


Loading MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Entailment ID: 0

Sanity check:
  'positive about the plot' → 0.9849
  'negative about the characters' → 0.9692
  'positive about the characters' → 0.0004


In [ ]:
# plot extraction

asp_name = "plot"
hyps = ASPECT_HYPOTHESES[asp_name]
pos_path = OUT_DIR / f"_raw_pos_{asp_name}.npy"
neg_path = OUT_DIR / f"_raw_neg_{asp_name}.npy"
 
if pos_path.exists() and neg_path.exists():
    print(f"{asp_name}: loading from checkpoint...")
    pos_plot = np.load(pos_path)
    neg_plot = np.load(neg_path)
else:
    print(f"Extracting {asp_name}... ({n:,} reviews)")
    pos_plot = np.zeros(n, dtype=np.float32)
    neg_plot = np.zeros(n, dtype=np.float32)
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} pos"):
        e = min(s + BATCH_SIZE, n)
        pos_plot[s:e] = score_batch(texts[s:e], hyps["positive"])
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} neg"):
        e = min(s + BATCH_SIZE, n)
        neg_plot[s:e] = score_batch(texts[s:e], hyps["negative"])
 
    np.save(pos_path, pos_plot)
    np.save(neg_path, neg_plot)
    print("Checkpoint saved!")
 
mask = (np.maximum(pos_plot, neg_plot) >= CONFIDENCE_THRESHOLD) & (np.abs(pos_plot - neg_plot) >= MARGIN_THRESHOLD)
print(f"\n{asp_name}: coverage {mask.sum():,}/{n:,} ({100*mask.sum()/n:.1f}%)")
print(f"  pos mean: {pos_plot.mean():.3f} | neg mean: {neg_plot.mean():.3f}")
polarity = pos_plot - neg_plot
valid = polarity[mask]
if len(valid) > 0:
    print(f"  polarity mean: {valid.mean():+.3f} | std: {valid.std():.3f}")

In [ ]:
# characters extraction
 
asp_name = "characters"
hyps = ASPECT_HYPOTHESES[asp_name]
pos_path = OUT_DIR / f"_raw_pos_{asp_name}.npy"
neg_path = OUT_DIR / f"_raw_neg_{asp_name}.npy"
 
if pos_path.exists() and neg_path.exists():
    print(f"{asp_name}: loading from checkpoint...")
    pos_chars = np.load(pos_path)
    neg_chars = np.load(neg_path)
else:
    print(f"Extracting {asp_name}... ({n:,} reviews)")
    pos_chars = np.zeros(n, dtype=np.float32)
    neg_chars = np.zeros(n, dtype=np.float32)
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} pos"):
        e = min(s + BATCH_SIZE, n)
        pos_chars[s:e] = score_batch(texts[s:e], hyps["positive"])
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} neg"):
        e = min(s + BATCH_SIZE, n)
        neg_chars[s:e] = score_batch(texts[s:e], hyps["negative"])
 
    np.save(pos_path, pos_chars)
    np.save(neg_path, neg_chars)
    print("Checkpoint saved!")
 
mask = (np.maximum(pos_chars, neg_chars) >= CONFIDENCE_THRESHOLD) & (np.abs(pos_chars - neg_chars) >= MARGIN_THRESHOLD)
print(f"\n{asp_name}: coverage {mask.sum():,}/{n:,} ({100*mask.sum()/n:.1f}%)")
print(f"  pos mean: {pos_chars.mean():.3f} | neg mean: {neg_chars.mean():.3f}")
polarity = pos_chars - neg_chars
valid = polarity[mask]
if len(valid) > 0:
    print(f"  polarity mean: {valid.mean():+.3f} | std: {valid.std():.3f}")

In [ ]:
# writing extraction

asp_name = "writing"
hyps = ASPECT_HYPOTHESES[asp_name]
pos_path = OUT_DIR / f"_raw_pos_{asp_name}.npy"
neg_path = OUT_DIR / f"_raw_neg_{asp_name}.npy"
 
if pos_path.exists() and neg_path.exists():
    print(f"{asp_name}: loading from checkpoint...")
    pos_writ = np.load(pos_path)
    neg_writ = np.load(neg_path)
else:
    print(f"Extracting {asp_name}... ({n:,} reviews)")
    pos_writ = np.zeros(n, dtype=np.float32)
    neg_writ = np.zeros(n, dtype=np.float32)
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} pos"):
        e = min(s + BATCH_SIZE, n)
        pos_writ[s:e] = score_batch(texts[s:e], hyps["positive"])
 
    for s in tqdm(range(0, n, BATCH_SIZE), desc=f"{asp_name} neg"):
        e = min(s + BATCH_SIZE, n)
        neg_writ[s:e] = score_batch(texts[s:e], hyps["negative"])
 
    np.save(pos_path, pos_writ)
    np.save(neg_path, neg_writ)
    print("Checkpoint saved!")
 
mask = (np.maximum(pos_writ, neg_writ) >= CONFIDENCE_THRESHOLD) & (np.abs(pos_writ - neg_writ) >= MARGIN_THRESHOLD)
print(f"\n{asp_name}: coverage {mask.sum():,}/{n:,} ({100*mask.sum()/n:.1f}%)")
print(f"  pos mean: {pos_writ.mean():.3f} | neg mean: {neg_writ.mean():.3f}")
polarity = pos_writ - neg_writ
valid = polarity[mask]
if len(valid) > 0:
    print(f"  polarity mean: {valid.mean():+.3f} | std: {valid.std():.3f}")
 
torch.cuda.empty_cache()

In [ ]:
for name in [
    "_raw_pos_plot.npy",
    "_raw_neg_plot.npy",
    "_raw_pos_characters.npy",
    "_raw_neg_characters.npy",
    "_raw_pos_writing.npy",
    "_raw_neg_writing.npy",
]:
    matches = list(Path("/kaggle/input").rglob(name))
    print(name, "->", matches[:3])

In [ ]:
# threshold, centering, item aggregation

from pathlib import Path
import numpy as np
import pandas as pd

SEARCH_DIRS = [
    OUT_DIR,                 
    DATA_PATH.parent,        
    Path("/kaggle/input"),   
]

def find_checkpoint(filename: str) -> Path:
    for root in SEARCH_DIRS:
        matches = list(root.rglob(filename))
        if matches:
            print(f"Loading {filename} from: {matches[0]}")
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} in:\n" + "\n".join(str(p) for p in SEARCH_DIRS)
    )

def load_checkpoint(filename: str, expected_len: int | None = None) -> np.ndarray:
    path = find_checkpoint(filename)
    arr = np.load(path)
    if expected_len is not None and len(arr) != expected_len:
        raise ValueError(
            f"{filename}: expected length {expected_len}, got {len(arr)} from {path}"
        )
    return arr

raw_pos = {
    "plot": load_checkpoint("_raw_pos_plot.npy", expected_len=n),
    "characters": load_checkpoint("_raw_pos_characters.npy", expected_len=n),
    "writing": load_checkpoint("_raw_pos_writing.npy", expected_len=n),
}

raw_neg = {
    "plot": load_checkpoint("_raw_neg_plot.npy", expected_len=n),
    "characters": load_checkpoint("_raw_neg_characters.npy", expected_len=n),
    "writing": load_checkpoint("_raw_neg_writing.npy", expected_len=n),
}

print(f"Applying threshold: conf >= {CONFIDENCE_THRESHOLD}, margin >= {MARGIN_THRESHOLD}\n")

review_raw = pd.DataFrame(index=valid_idx)
for asp_name in ASP_NAMES:
    col = f"nli_aspect_{asp_name}"
    pos = raw_pos[asp_name].copy()
    neg = raw_neg[asp_name].copy()

    confident = (np.maximum(pos, neg) >= CONFIDENCE_THRESHOLD) & (np.abs(pos - neg) >= MARGIN_THRESHOLD)
    polarity = pos - neg
    polarity[~confident] = np.nan

    review_raw[col] = polarity
    n_conf = confident.sum()
    mean_val = polarity[confident].mean() if n_conf > 0 else np.nan
    print(f"  {asp_name}: coverage {n_conf:,}/{n:,} ({100*n_conf/n:.1f}%) | mean={mean_val:+.3f}")

print(f"\nCentering...")
review_centered = review_raw.copy()
n_valid = review_centered[ASP_COLS].notna().sum(axis=1)
review_mean = review_centered[ASP_COLS].mean(axis=1, skipna=True)
center_mask = n_valid >= 2

for col in ASP_COLS:
    review_centered.loc[center_mask, col] = review_centered.loc[center_mask, col] - review_mean.loc[center_mask]

print(f"  Centered {center_mask.sum():,}/{n:,} reviews ({100*center_mask.sum()/n:.1f}%)")
print(f"  Skipped {(~center_mask).sum():,} (0 or 1 valid aspect)")

train_df = df.loc[valid_idx].copy()

RAW_COLS = [f"nli_raw_{asp}" for asp in ASP_NAMES]
CTR_COLS = [f"nli_ctr_{asp}" for asp in ASP_NAMES]

for asp, raw_col, ctr_col in zip(ASP_NAMES, RAW_COLS, CTR_COLS):
    src_col = f"nli_aspect_{asp}"
    train_df[raw_col] = review_raw[src_col].values
    train_df[ctr_col] = review_centered[src_col].values

ALL_NLI_COLS = RAW_COLS + CTR_COLS
item_aspects = train_df.groupby("item_id")[ALL_NLI_COLS].mean()

item_counts = train_df.groupby("item_id").agg(
    n_reviews=("item_id", "size"),
    n_plot=("nli_raw_plot", lambda s: s.notna().sum()),
    n_characters=("nli_raw_characters", lambda s: s.notna().sum()),
    n_writing=("nli_raw_writing", lambda s: s.notna().sum()),
)
item_aspects = item_aspects.join(item_counts)

n_all_items = df["item_id"].nunique()
print(f"\nItem profiles: {len(item_aspects):,} / {n_all_items:,}")

In [ ]:
# stats, correlations
 
print("="*60)
print("RAW NLI ASPECTS (item-level)")
print("="*60)
for col in RAW_COLS:
    vals = item_aspects[col].dropna()
    print(f"  {col:<25} mean={vals.mean():+.3f}  std={vals.std():.3f}  n={len(vals):,}")
 
print(f"\nRAW inter-aspect correlations:")
raw_corr = item_aspects[RAW_COLS].corr()
for i, c1 in enumerate(RAW_COLS):
    for j, c2 in enumerate(RAW_COLS):
        if j > i:
            print(f"  {c1.replace('nli_raw_','')} vs {c2.replace('nli_raw_','')}: r={raw_corr.loc[c1,c2]:.3f}")
 
print(f"\n{'='*60}")
print("CENTERED NLI ASPECTS (item-level)")
print("="*60)
for col in CTR_COLS:
    vals = item_aspects[col].dropna()
    print(f"  {col:<25} mean={vals.mean():+.3f}  std={vals.std():.3f}  n={len(vals):,}")
 
print(f"\nCENTERED inter-aspect correlations:")
ctr_corr = item_aspects[CTR_COLS].corr()
for i, c1 in enumerate(CTR_COLS):
    for j, c2 in enumerate(CTR_COLS):
        if j > i:
            print(f"  {c1.replace('nli_ctr_','')} vs {c2.replace('nli_ctr_','')}: r={ctr_corr.loc[c1,c2]:.3f}")
 
old_cols = ["aspect_plot", "aspect_characters", "aspect_writing"]
if all(c in df.columns for c in old_cols):
    print(f"\n{'='*60}")
    print("OLD (similarity-based) correlations")
    print("="*60)
    old_item = df.groupby("item_id")[old_cols].mean()
    old_corr = old_item.corr()
    for i, c1 in enumerate(old_cols):
        for j, c2 in enumerate(old_cols):
            if j > i:
                print(f"  {c1} vs {c2}: r={old_corr.loc[c1,c2]:.3f}")
 
if RATING_COL:
    for col in ALL_NLI_COLS:
        if col in df.columns:
            df = df.drop(columns=[col])
    df = df.join(item_aspects[ALL_NLI_COLS], on="item_id", how="left")
 
    print(f"\n{'='*60}")
    print("POLARITY vs RATING (item-level)")
    print("="*60)
    for col in RAW_COLS:
        means = [df.loc[df[RATING_COL]==r, col].mean() for r in [1,2,3,4,5]]
        print(f"  {col}: " + " | ".join(f"r{r}={m:+.3f}" for r,m in zip([1,2,3,4,5], means)))
 
if RATING_COL and RATING_COL in train_df.columns:
    print(f"\n{'='*60}")
    print("POLARITY vs RATING (review-level)")
    print("="*60)
    for col in RAW_COLS:
        means = [train_df.loc[train_df[RATING_COL]==r, col].mean() for r in [1,2,3,4,5]]
        print(f"  {col}: " + " | ".join(f"r{r}={m:+.3f}" for r,m in zip([1,2,3,4,5], means)))
 

In [ ]:
# graphs
 
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
 
for i, (asp, raw_col, ctr_col) in enumerate(zip(ASP_NAMES, RAW_COLS, CTR_COLS)):
    vals = item_aspects[raw_col].dropna()
    axes[0, i].hist(vals, bins=50, edgecolor="black", alpha=0.7, color="#4A90D9")
    axes[0, i].set_title(f"{asp.title()} (raw)")
    axes[0, i].axvline(0, color="red", linestyle="--", alpha=0.5)
 
    vals = item_aspects[ctr_col].dropna()
    axes[1, i].hist(vals, bins=50, edgecolor="black", alpha=0.7, color="#50C878")
    axes[1, i].set_title(f"{asp.title()} (centered)")
    axes[1, i].axvline(0, color="red", linestyle="--", alpha=0.5)
 
plt.suptitle("NLI Aspect Distributions: Raw vs Centered (item-level)", fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / "nli_distributions_raw_vs_centered.png", dpi=150, bbox_inches="tight")
plt.show()
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
labels = [a.title() for a in ASP_NAMES]
 
sns.heatmap(raw_corr.values, annot=True, fmt=".3f", cmap="coolwarm",
            xticklabels=labels, yticklabels=labels, center=0, ax=ax1, vmin=-1, vmax=1)
ax1.set_title("Raw correlations")
 
sns.heatmap(ctr_corr.values, annot=True, fmt=".3f", cmap="coolwarm",
            xticklabels=labels, yticklabels=labels, center=0, ax=ax2, vmin=-1, vmax=1)
ax2.set_title("Centered correlations")
 
plt.suptitle("Inter-Aspect Correlations (item-level)", fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / "nli_correlations_raw_vs_centered.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
item_aspects.reset_index().to_parquet(OUT_DIR / "item_nli_aspects.parquet", index=False)
print(f"Saved: item_nli_aspects.parquet ({len(item_aspects):,} items, {len(ALL_NLI_COLS)} cols)")
 
df.to_parquet(OUT_DIR / "books_big_nli.parquet", index=False)
print(f"Saved: books_big_nli.parquet ({len(df):,} rows)")
 
review_base = ["user_id", "item_id", "timestamp", "text_combined"]
if RATING_COL and RATING_COL in train_df.columns:
    review_base.append(RATING_COL)
review_out = train_df[review_base + ALL_NLI_COLS].copy()
review_out.to_parquet(OUT_DIR / "review_nli_aspects.parquet", index=False)
print(f"Saved: review_nli_aspects.parquet ({len(review_out):,} reviews)")
 
extreme_rows = []
for col in RAW_COLS:
    valid = review_out.dropna(subset=[col])
    if len(valid) == 0:
        continue
    extreme_rows.append(valid.nlargest(20, col).assign(aspect=col, direction="top_positive"))
    extreme_rows.append(valid.nsmallest(20, col).assign(aspect=col, direction="top_negative"))
if extreme_rows:
    pd.concat(extreme_rows, ignore_index=True).to_csv(OUT_DIR / "nli_extreme_examples.csv", index=False)
    print(f"Saved: nli_extreme_examples.csv")
 
print(f"\nAll files in {OUT_DIR}:")
for f in sorted(OUT_DIR.glob("*nli*")):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name} ({size_mb:.1f} MB)")
 
print(f"Output columns: {ALL_NLI_COLS}")
 